In [1]:
# Core imports
import os, sys, asyncio, yaml
from dotenv import load_dotenv
from pydantic import BaseModel
from contextlib import asynccontextmanager
# Project modules (assumes this notebook lives in src/mcp_client/)
from client import MCPClient
import json
import requests

In [2]:
import logging
import sys
from typing import Optional

DEFAULT_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

def get_logger(
    name: str,
    log_file: Optional[str] = None,
    level: int = logging.INFO,
    console_level: Optional[int] = None,
    fmt: str = DEFAULT_FORMAT,
    propagate: bool = False,
):
    """Return a configured logger.

    Ensures we don't attach duplicate handlers if called multiple times.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = propagate

    formatter = logging.Formatter(fmt)

    # Add/ensure file handler
    if log_file:
        if not any(isinstance(h, logging.FileHandler) and getattr(h, 'baseFilename', None) and h.baseFilename.endswith(log_file) for h in logger.handlers):
            fh = logging.FileHandler(log_file)
            fh.setLevel(level)
            fh.setFormatter(formatter)
            logger.addHandler(fh)

    # Add/ensure console handler
    if console_level is None:
        console_level = level
    if not any(isinstance(h, logging.StreamHandler) and getattr(h, 'stream', None) is sys.stdout for h in logger.handlers):
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(console_level)
        ch.setFormatter(formatter)
        logger.addHandler(ch)

    return logger

In [3]:
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY", "")

logger = get_logger("mcp-client", log_file="mcp_client.log", level=20, console_level=20)

from pathlib import Path
file_path = Path("/home/vmadmin/intent/src/test/anthropic_teste.csv")

In [4]:
# Create a client, connect, and process an intent (returns an OpenAI Responses API object)
client = MCPClient(logger=logger, rapp="http://10.233.7.113:8090", file_path= file_path)
connected = await client.connect_to_server("http://127.0.0.1:8000/sse")
await client.set_llm(api_key=api_key, llm_name="qwen",llm_model="qwen/qwen3-235b-a22b")
if not connected:
    raise RuntimeError("Failed to connect to MCP server")

# # 'response' is an object (not a dict) with an 'output' attribute (a list of bl1ocks)
response = await client.process_intent(
   "Provision a slice for industrial motion control networks, requiring latency not exceeding 2 ms and uplink rates of 15 Mbps."
)

2026-09-19 17:14:04,497 - mcp-client - INFO - Attempting to connect to server at http://127.0.0.1:8000/sse.
2026-09-19 17:14:04,614 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-19 17:14:04,619 - mcp-client - INFO - Successfully connected to server. Available tools: ['create_session', 'get_session', 'delete_session', 'ping_api']
2026-09-19 17:14:04,620 - mcp-client - INFO - Setting llm qwen - model qwen/qwen3-235b-a22b
2026-09-19 17:14:04,620 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-19 17:14:04,641 - mcp-client - INFO - Setting llm qwen - tool {'$defs': {'Area': {'properties': {'areaType': {'anyOf': [{'$ref': '#/$defs/AreaType'}, {'type': 'null'}], 'default': None}}, 'type': 'object'}, 'AreaType': {'enum': ['CIRCLE', 'POLYGON'], 'type': 'string'}, 'CreateSession': {'properties': {'serviceTime': {'anyOf': [{'$ref': '#/$defs/TimePeriod'}, {'type': 'null'}], 'description': 'serviceTime is a period during which the network slice will be rese

Traceback (most recent call last):
  File "/home/vmadmin/intent/src/mcp_client/client.py", line 451, in call_qwen
    )
      
  File "/usr/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 353, in raw_decode
    obj, end = self.scan_once(s, idx)
               ^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 1 column 279 (char 278)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3506, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1826782/409384576